# Make Splits Notebook

- Source: `scripts/make_splits.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""
데이터 스플릿 스크립트.

data/raw/<task>/.../<class>/*.jpg 를 스캔해서
data/annotations/<task>.csv 와 data/splits/<task>_{train,val,test}.csv 생성.

스플릿 비율 : train 70% / val 15% / test 15% (stratified → 클래스 비율 유지)

Kaggle 데이터는 'session_id' 개념이 없으므로 GroupSplit은 사용하지 않음.

사용 예시 (중요!)
───────────────
    # 로스팅 4클래스 (250장씩 다운샘플)
    python scripts/make_splits.py --task roast --max_samples 1000

    # 결점두 17클래스 (6000장 전체 사용)
    python scripts/make_splits.py --task defect --classes broken cut "dry cherry" fade floater "full black" "full sour" "fungus damange" husk immature parchment "partial black" "partial sour" "severe insect damange" shell "slight insect damage" withered

    # 결점두 6000장 전체 사용 (--max_samples 없음 = 제한 없음)
    # 이렇게 하면 클래스별 수 차이가 난다 → class weight가 자동 보정해줌
"""
from __future__ import annotations
import argparse
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


## Step 2. Function: find_class_dir

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def find_class_dir(raw_root: Path, cls: str) -> list[Path]:
    """raw_root 하위에서 폴더명이 cls(대소문자 무시)인 디렉토리를 모두 찾는다."""
    matches = []
    for p in raw_root.rglob("*"):
        # 폴더명만 기준으로 찾기 때문에 원본 데이터 구조가 조금 달라도 대응 가능하다.
        if p.is_dir() and p.name.lower() == cls.lower():
            matches.append(p)
    return matches


## Step 3. Function: collect_images

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def collect_images(raw_root: Path, classes: list[str], task: str) -> pd.DataFrame:
    rows = []
    for cls in classes:
        dirs = find_class_dir(raw_root, cls)
        if not dirs:
            print(f"[!] class '{cls}' 폴더를 {raw_root} 아래에서 찾지 못함")
            continue
        for d in dirs:
            for img in d.rglob("*"):
                if img.suffix.lower() in IMG_EXT and img.is_file():
                    # source는 나중에 어떤 데이터 소스/하위 폴더에서 왔는지
                    # 추적할 때 도움이 되도록 같이 저장한다.
                    rows.append({
                        "path": img.resolve().as_posix(),
                        f"{task}_label": cls.lower(),
                        "source": d.relative_to(raw_root).parts[0]
                                  if d != raw_root else "root",
                    })
    df = pd.DataFrame(rows)
    return df


## Step 4. Function: stratified_split

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def stratified_split(df: pd.DataFrame, label_col: str, seed: int = 42):
    # 70 / 15 / 15 비율.
    # stratify를 써서 각 split에 클래스 비율이 최대한 유지되게 만든다.
    train, temp = train_test_split(df, test_size=0.30, stratify=df[label_col],
                                   random_state=seed)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp[label_col],
                                 random_state=seed)
    return train, val, test


## Step 5. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--task", required=True, choices=["roast", "defect"])
    p.add_argument("--raw", default="data/raw")
    p.add_argument("--out_ann", default="data/annotations")
    p.add_argument("--out_split", default="data/splits")
    p.add_argument("--classes", nargs="+", default=None,
                   help="클래스 목록. 미지정 시 task별 기본값 사용")
    p.add_argument("--max_samples", type=int, default=None,
                   help="총 샘플 수 상한. stratified 다운샘플링.")
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()

    default_classes = {
        "roast":  ["green", "light", "medium", "dark"],
        "defect": ["normal", "defective"],
    }
    classes = args.classes or default_classes[args.task]
    raw_root = Path(args.raw) / args.task

    print(f"[+] scan: {raw_root}  classes={classes}")
    df = collect_images(raw_root, classes, args.task)
    if df.empty:
        raise SystemExit("[X] 수집된 이미지가 없습니다. 데이터 경로/클래스명을 확인하세요.")

    print("[+] class distribution:")
    print(df[f"{args.task}_label"].value_counts())

    if args.max_samples and len(df) > args.max_samples:
        n = args.max_samples
        frac = n / len(df)
        parts = []
        for _, g in df.groupby(f"{args.task}_label"):
            # downsample도 클래스 비율을 해치지 않도록 그룹별로 나눠서 뽑는다.
            k = max(1, int(round(len(g) * frac)))
            parts.append(g.sample(k, random_state=args.seed))
        df = pd.concat(parts, ignore_index=True)
        print(f"[+] downsampled to {len(df)} rows (target {n})")
        print(df[f"{args.task}_label"].value_counts())

    Path(args.out_ann).mkdir(parents=True, exist_ok=True)
    Path(args.out_split).mkdir(parents=True, exist_ok=True)

    ann_path = Path(args.out_ann) / f"{args.task}.csv"
    df.to_csv(ann_path, index=False)
    print(f"[OK] annotation -> {ann_path} ({len(df)} rows)")

    train, val, test = stratified_split(df, f"{args.task}_label", args.seed)
    for name, part in [("train", train), ("val", val), ("test", test)]:
        out = Path(args.out_split) / f"{args.task}_{name}.csv"
        part.to_csv(out, index=False)
        print(f"[OK] {name:5s} -> {out} ({len(part)} rows)")


## Step 6. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
